# 🚀 NEXUS Stock AI — Phase 1: High-Performance FNSPID Dataset Ingestion

**Goal:** Ingest, filter, and persist the FNSPID Financial News (23.2 GB raw CSV) and Historical Stock Prices dataset for **10 MVP Tickers** on Google Colab (T4 GPU Runtime) without triggering OOM crashes or running out of disk space.

### 🎯 10 MVP Tickers
`AAPL`, `MSFT`, `NVDA`, `AMZN`, `GOOGL`, `META`, `TSLA`, `AMD`, `JPM`, `NFLX`

### ⚡ Key Optimizations Implemented
1. **High-Speed Multi-Threaded Download:** Utilizes `aria2c` with 16 parallel connections to download at 100–200 MB/s, saturating Colab's network.
2. **C-Engine Chunked Streaming for News (23.2 GB):** Uses Pandas' optimized C engine with `chunksize=250,000` and `on_bad_lines='skip'` to bypass known parsing errors without loading the whole file into RAM.
3. **Zero RAM Accumulation with PyArrow ParquetWriter:** Filtered rows are written incrementally as row groups using PyArrow with `zstd` compression. Memory footprint stays under **~1.5 GB**.
4. **In-Memory Selective Extraction for Prices:** Reads only the 10 target ticker CSVs directly from `full_history.zip` via `zipfile.ZipFile`, avoiding the disk and inode overhead of extracting 6,000+ files.
5. **Strict Disk and RAM Management:** Immediately deletes raw CSV and ZIP files after parquet generation and triggers `gc.collect()`.
6. **DuckDB Out-of-Core Validation:** Validates row counts, date ranges, and missing values using zero-memory-copy SQL queries directly against the parquet files.

## ⚙️ Step 1: Environment Setup & Optimization
Check system hardware specs (RAM, Disk, GPU) and install high-performance I/O dependencies.

In [1]:
# 1. Install high-speed download utility (aria2) and data processing libraries
!apt-get update -qq && apt-get install -y -qq aria2
%pip install -q pyarrow fastparquet duckdb tqdm

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libcares2:amd64.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.37.0+debian-1build3_amd64.deb ...
Unpacking libaria2-0:amd64 (1.37.0+debian-1build3) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.37.0+debian-1build3_amd64.deb ...
Unpacking aria2 (1.37.0+debian-1build3) ...
Setting up l

In [2]:
import os
import sys
import gc
import time
import shutil
import zipfile
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
from tqdm.auto import tqdm

# --- Global Configuration ---
MVP_TICKERS = {"AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA", "AMD", "JPM", "NFLX"}

DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

RAW_NEWS_CSV = os.path.join(DATA_DIR, "nasdaq_exteral_data.csv")
RAW_PRICES_ZIP = os.path.join(DATA_DIR, "full_history.zip")

OUTPUT_NEWS_PARQUET = os.path.join(DATA_DIR, "filtered_news.parquet")
OUTPUT_PRICES_PARQUET = os.path.join(DATA_DIR, "filtered_prices.parquet")

NEWS_URL = "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/nasdaq_exteral_data.csv"
PRICES_URL = "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_price/full_history.zip"

def print_system_stats(stage_name=""):
    """Utility to display current RAM and Disk usage."""
    total, used, free = shutil.disk_usage("/")
    disk_used_gb = used / (1024**3)
    disk_total_gb = total / (1024**3)
    disk_free_gb = free / (1024**3)
    
    mem_info = ""
    try:
        import psutil
        vm = psutil.virtual_memory()
        mem_used_gb = vm.used / (1024**3)
        mem_total_gb = vm.total / (1024**3)
        mem_info = f" | RAM: {mem_used_gb:.2f}GB / {mem_total_gb:.2f}GB ({vm.percent}%)"
    except ImportError:
        pass
        
    print(f"[{stage_name}] Disk Used: {disk_used_gb:.2f}GB / {disk_total_gb:.2f}GB (Free: {disk_free_gb:.2f}GB){mem_info}")

print_system_stats("Initial Hardware Stats")

[Initial Hardware Stats] Disk Used: 47.44GB / 112.64GB (Free: 65.18GB) | RAM: 0.75GB / 12.67GB (8.4%)


## 📥 Step 2: High-Bandwidth Dataset Downloads
We use `aria2c` configured with 16 parallel connections to download directly from Hugging Face's CDN into Colab's local storage. This avoids single-threaded throttling and completes the download in minutes.

In [3]:
def fast_download(url: str, dest_path: str):
    """Downloads a file using aria2c (16 parallel connections) with curl fallback."""
    if os.path.exists(dest_path):
        size_gb = os.path.getsize(dest_path) / 1e9
        print(f"File already exists: {dest_path} ({size_gb:.2f} GB). Skipping download.")
        return
        
    dest_dir = os.path.dirname(dest_path)
    filename = os.path.basename(dest_path)
    print(f"\nStarting download: {filename} from {url}")
    start_time = time.time()
    
    # Run aria2c with multi-connection acceleration
    cmd = f'aria2c -x 16 -s 16 -j 16 -k 1M --continue=true --file-allocation=none --dir="{dest_dir}" --out="{filename}" "{url}"'
    ret = os.system(cmd)
    
    if ret != 0 or not os.path.exists(dest_path):
        print("aria2c encountered an issue; falling back to curl...")
        os.system(f'curl -L -C - -o "{dest_path}" "{url}"')
        
    elapsed = time.time() - start_time
    if os.path.exists(dest_path):
        size_gb = os.path.getsize(dest_path) / 1e9
        print(f"Finished download: {filename} ({size_gb:.2f} GB) in {elapsed:.1f}s ({size_gb*1024/max(elapsed, 1):.1f} MB/s)")
    else:
        raise FileNotFoundError(f"Failed to download {dest_path}")

print_system_stats("Pre-Download")
# Download both raw datasets
fast_download(PRICES_URL, RAW_PRICES_ZIP)
fast_download(NEWS_URL, RAW_NEWS_CSV)
print_system_stats("Post-Download")

[Pre-Download] Disk Used: 47.44GB / 112.64GB (Free: 65.18GB) | RAM: 0.75GB / 12.67GB (8.4%)

Starting download: full_history.zip from https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_price/full_history.zip
Finished download: full_history.zip (0.59 GB) in 3.6s (169.4 MB/s)

Starting download: nasdaq_exteral_data.csv from https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/nasdaq_exteral_data.csv
Finished download: nasdaq_exteral_data.csv (23.23 GB) in 143.8s (165.4 MB/s)
[Post-Download] Disk Used: 69.63GB / 112.64GB (Free: 42.99GB) | RAM: 0.82GB / 12.67GB (8.9%)


## 📰 Step 3: News Data Processing (23.2 GB CSV $\rightarrow$ `filtered_news.parquet`)

### Memory & I/O Strategy:
- **Chunk Size:** 250,000 rows (~250–350 MB in memory per chunk).
- **C Engine & Error Handling:** `engine='c'` with `on_bad_lines='skip'` reliably bypasses known multiline and delimiter errors in FNSPID without aborting.
- **PyArrow Incremental ParquetWriter:** Filtered rows are converted into Arrow tables matching a unified schema and appended directly to `filtered_news.parquet` using `zstd` level 7 compression. Chunks are discarded from memory immediately.
- **Strict Cleanup:** Immediately removes the 23.2 GB raw CSV file upon completion and triggers `gc.collect()`.

In [4]:
def process_news_data(
    input_csv: str = RAW_NEWS_CSV,
    output_parquet: str = OUTPUT_NEWS_PARQUET,
    target_tickers: set = MVP_TICKERS,
    chunksize: int = 250_000
):
    """
    Processes the 23.2 GB FNSPID news dataset in chunks,
    filters for MVP tickers, saves to compressed Parquet,
    and strictly deletes raw CSV upon completion.
    """
    if not os.path.exists(input_csv):
        print(f"Raw CSV {input_csv} not found. Skipping or already processed.")
        return
        
    print("=" * 70)
    print("STREAMING & FILTERING FNSPID NEWS (23.2 GB CSV)")
    print(f"Target Tickers ({len(target_tickers)}): {sorted(list(target_tickers))}")
    print(f"Chunk Size: {chunksize:,} rows | Compression: ZSTD (lvl 7)")
    print("=" * 70)
    
    start_time = time.time()
    total_raw_rows = 0
    total_matched_rows = 0
    writer = None
    
    # Unified Arrow schema to ensure zero type-mismatch across chunks
    NEWS_ARROW_SCHEMA = pa.schema([
        pa.field('Date', pa.string()),
        pa.field('Stock_symbol', pa.string()),
        pa.field('Article_title', pa.string()),
        pa.field('Url', pa.string()),
        pa.field('Publisher', pa.string()),
        pa.field('Author', pa.string()),
        pa.field('Article', pa.string()),
        pa.field('Lsa_summary', pa.string()),
        pa.field('Luhn_summary', pa.string()),
        pa.field('Textrank_summary', pa.string()),
        pa.field('Lexrank_summary', pa.string()),
    ])
    
    string_cols = NEWS_ARROW_SCHEMA.names
    
    # Temporary output path for atomicity
    temp_output = output_parquet + ".tmp"
    if os.path.exists(temp_output):
        os.remove(temp_output)
    if os.path.exists(output_parquet):
        os.remove(output_parquet)
        
    chunk_idx = 0
    pbar = tqdm(desc="News Ingestion Chunks", unit="chunk")
    
    try:
        # Use C-engine with on_bad_lines='skip' for fast parsing and error recovery
        csv_reader = pd.read_csv(
            input_csv,
            chunksize=chunksize,
            engine='c',
            on_bad_lines='skip',
            low_memory=False
        )
        
        for chunk in csv_reader:
            chunk_idx += 1
            total_raw_rows += len(chunk)
            
            # Strip column whitespace
            chunk.columns = [c.strip() for c in chunk.columns]
            
            if 'Stock_symbol' not in chunk.columns:
                pbar.update(1)
                continue
                
            # Normalize ticker symbol column
            chunk['Stock_symbol'] = chunk['Stock_symbol'].astype(str).str.strip().str.upper()
            
            # Fast boolean indexing filter for 10 MVP tickers
            filtered_chunk = chunk[chunk['Stock_symbol'].isin(target_tickers)].copy()
            matched_count = len(filtered_chunk)
            total_matched_rows += matched_count
            
            if matched_count > 0:
                # Align missing columns and cast to clean strings
                for col in string_cols:
                    if col not in filtered_chunk.columns:
                        filtered_chunk[col] = ""
                    else:
                        filtered_chunk[col] = filtered_chunk[col].fillna("").astype(str)
                
                filtered_chunk = filtered_chunk[string_cols]
                
                # Convert to Arrow Table and append to Parquet
                table = pa.Table.from_pandas(filtered_chunk, schema=NEWS_ARROW_SCHEMA, preserve_index=False)
                
                if writer is None:
                    writer = pq.ParquetWriter(
                        temp_output,
                        NEWS_ARROW_SCHEMA,
                        compression='zstd',
                        compression_level=7
                    )
                writer.write_table(table)
                
            pbar.update(1)
            pbar.set_postfix({
                "raw_rows": f"{total_raw_rows:,}",
                "matched": f"{total_matched_rows:,}"
            })
            
            # Explicit memory release per chunk
            del chunk, filtered_chunk
            if chunk_idx % 10 == 0:
                gc.collect()
                
    finally:
        pbar.close()
        if writer is not None:
            writer.close()
            
    if os.path.exists(temp_output):
        os.rename(temp_output, output_parquet)
        
    elapsed = time.time() - start_time
    parquet_size_mb = os.path.getsize(output_parquet) / (1024 * 1024) if os.path.exists(output_parquet) else 0
    
    print("-" * 70)
    print(f"Processing finished in {elapsed:.1f}s ({elapsed/60:.2f} mins)")
    print(f"Total Raw Rows:          {total_raw_rows:,}")
    print(f"Total Filtered MVP Rows: {total_matched_rows:,} ({total_matched_rows/max(total_raw_rows,1)*100:.2f}%)")
    print(f"Saved Final Parquet:     {output_parquet} ({parquet_size_mb:.2f} MB)")
    print("-" * 70)
    
    # --- STRICT DISK CLEANUP ---
    if os.path.exists(input_csv):
        raw_size_gb = os.path.getsize(input_csv) / 1e9
        print(f"🗑️ Deleting raw CSV {input_csv} ({raw_size_gb:.2f} GB) to reclaim disk...")
        os.remove(input_csv)
        print("Raw CSV deleted successfully.")
        
    gc.collect()
    print_system_stats("Post-News Processing")

# Execute news processing
process_news_data()

STREAMING & FILTERING FNSPID NEWS (23.2 GB CSV)
Target Tickers (10): ['AAPL', 'AMD', 'AMZN', 'GOOGL', 'JPM', 'META', 'MSFT', 'NFLX', 'NVDA', 'TSLA']
Chunk Size: 250,000 rows | Compression: ZSTD (lvl 7)


News Ingestion Chunks: 0chunk [00:00, ?chunk/s]

----------------------------------------------------------------------
Processing finished in 418.2s (6.97 mins)
Total Raw Rows:          15,549,299
Total Filtered MVP Rows: 62,458 (0.40%)
Saved Final Parquet:     ./data/filtered_news.parquet (78.43 MB)
----------------------------------------------------------------------
🗑️ Deleting raw CSV ./data/nasdaq_exteral_data.csv (23.23 GB) to reclaim disk...
Raw CSV deleted successfully.
[Post-News Processing] Disk Used: 69.71GB / 112.64GB (Free: 42.92GB) | RAM: 2.67GB / 12.67GB (23.5%)


## 📈 Step 4: Price Data Processing (`full_history.zip` $\rightarrow$ `filtered_prices.parquet`)

### In-Memory Selective Extraction:
- Instead of uncompressing 6,000+ files to disk, we open `full_history.zip` using Python's `zipfile.ZipFile`.
- We selectively stream and parse **only the 10 target MVP ticker CSVs** (`full_history/{TICKER}.csv`).
- Standardizes column names (`date`, `stock_symbol`, `open`, `high`, `low`, `close`, `adj_close`, `volume`).
- Exports the combined table to `filtered_prices.parquet` with ZSTD compression.
- Immediately deletes `full_history.zip` and triggers `gc.collect()`.

In [5]:
def process_prices_data(
    input_zip: str = RAW_PRICES_ZIP,
    output_parquet: str = OUTPUT_PRICES_PARQUET,
    target_tickers: set = MVP_TICKERS
):
    """
    Selectively extracts 10 MVP tickers in memory from full_history.zip,
    combines into a single unified Parquet dataset, and deletes raw ZIP.
    """
    if not os.path.exists(input_zip):
        print(f"Raw ZIP {input_zip} not found. Skipping or already processed.")
        return
        
    print("=" * 70)
    print("SELECTIVE IN-MEMORY EXTRACTION OF HISTORICAL PRICES")
    print(f"Target Tickers ({len(target_tickers)}): {sorted(list(target_tickers))}")
    print("=" * 70)
    
    start_time = time.time()
    ticker_dfs = []
    
    with zipfile.ZipFile(input_zip, 'r') as zf:
        namelist = set(zf.namelist())
        
        for ticker in sorted(list(target_tickers)):
            # Potential path formats in FNSPID ZIP archive
            candidates = [
                f"full_history/{ticker}.csv",
                f"{ticker}.csv",
                f"full_history/{ticker.lower()}.csv"
            ]
            
            target_member = next((c for c in candidates if c in namelist), None)
            
            if not target_member:
                print(f"⚠️ Warning: Ticker '{ticker}' was not found in the ZIP archive!")
                continue
                
            # Read CSV directly from memory buffer without extracting to disk
            with zf.open(target_member) as f:
                df = pd.read_csv(f)
                
                # Standardize column headers
                df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
                
                # Add ticker identifier
                df['stock_symbol'] = ticker
                
                # Format Date column
                if 'date' in df.columns:
                    df['date'] = pd.to_datetime(df['date'], errors='coerce')
                    df = df.dropna(subset=['date'])
                    df['date'] = df['date'].dt.strftime('%Y-%m-%d')
                    
                ticker_dfs.append(df)
                print(f"  ✓ Extracted {ticker:5s}: {len(df):6,d} rows | Range: {df['date'].min()} to {df['date'].max()}")
                
    if not ticker_dfs:
        raise ValueError("No ticker price data was extracted from ZIP.")
        
    # Concatenate and sort chronologically
    combined_prices = pd.concat(ticker_dfs, ignore_index=True)
    combined_prices = combined_prices.sort_values(by=['stock_symbol', 'date']).reset_index(drop=True)
    
    # Save using PyArrow with ZSTD compression
    combined_prices.to_parquet(
        output_parquet,
        engine='pyarrow',
        compression='zstd',
        index=False
    )
    
    elapsed = time.time() - start_time
    parquet_size_mb = os.path.getsize(output_parquet) / (1024 * 1024)
    
    print("-" * 70)
    print(f"Prices processing completed in {elapsed:.2f}s")
    print(f"Total Combined Price Rows: {len(combined_prices):,}")
    print(f"Saved Final Parquet:       {output_parquet} ({parquet_size_mb:.2f} MB)")
    print("-" * 70)
    
    # Free dataframe memory
    del ticker_dfs, combined_prices
    
    # --- STRICT DISK CLEANUP ---
    if os.path.exists(input_zip):
        zip_size_mb = os.path.getsize(input_zip) / (1024 * 1024)
        print(f"🗑️ Deleting raw prices ZIP {input_zip} ({zip_size_mb:.1f} MB)...")
        os.remove(input_zip)
        print("Raw prices ZIP deleted successfully.")
        
    gc.collect()
    print_system_stats("Post-Prices Processing")

# Execute prices processing
process_prices_data()

SELECTIVE IN-MEMORY EXTRACTION OF HISTORICAL PRICES
Target Tickers (10): ['AAPL', 'AMD', 'AMZN', 'GOOGL', 'JPM', 'META', 'MSFT', 'NFLX', 'NVDA', 'TSLA']
  ✓ Extracted AAPL : 10,852 rows | Range: 1980-12-12 to 2023-12-28
  ✓ Extracted AMD  : 11,040 rows | Range: 1980-03-17 to 2023-12-28
  ✓ Extracted AMZN :  6,700 rows | Range: 1997-05-15 to 2023-12-28
  ✓ Extracted GOOGL:  3,932 rows | Range: 2004-08-19 to 2020-04-01
  ✓ Extracted JPM  : 11,040 rows | Range: 1980-03-17 to 2023-12-28
⚠️ Warning: Ticker 'META' was not found in the ZIP archive!
  ✓ Extracted MSFT :  9,526 rows | Range: 1986-03-13 to 2023-12-28
  ✓ Extracted NFLX :  4,551 rows | Range: 2002-05-23 to 2020-06-19
  ✓ Extracted NVDA :  6,275 rows | Range: 1999-01-22 to 2023-12-28
  ✓ Extracted TSLA :  3,399 rows | Range: 2010-06-29 to 2023-12-28
----------------------------------------------------------------------
Prices processing completed in 0.80s
Total Combined Price Rows: 67,315
Saved Final Parquet:       ./data/filtered

## 📊 Step 5: Out-of-Core Validation & Statistics
We use **DuckDB** to execute lightning-fast analytical queries directly over `filtered_news.parquet` and `filtered_prices.parquet` without loading entire datasets into RAM.

This step inspects:
1. **Total row counts & distinct ticker counts**
2. **Global date boundaries (Earliest vs Latest date)**
3. **Per-ticker breakdown (row counts & date ranges)**
4. **Missing / Null value diagnostics per column**
5. **Data sample previews**

In [6]:
def run_validation_suite(news_parquet: str = OUTPUT_NEWS_PARQUET, prices_parquet: str = OUTPUT_PRICES_PARQUET):
    """Runs comprehensive validation and prints statistics for both datasets using DuckDB."""
    print("=" * 80)
    print("🎯 PHASE 1 COMPREHENSIVE VALIDATION & STATS REPORT")
    print("=" * 80)
    
    con = duckdb.connect()
    
    # ================= 1. NEWS DATASET VALIDATION =================
    print("\n" + "═" * 30 + " 1. FILTERED NEWS DATASET " + "═" * 30)
    if os.path.exists(news_parquet):
        file_size_mb = os.path.getsize(news_parquet) / (1024 * 1024)
        print(f"File: {news_parquet} | Size: {file_size_mb:.2f} MB\n")
        
        # High-level summary
        news_summary = con.execute(f"""
            SELECT 
                COUNT(*) AS total_rows,
                COUNT(DISTINCT Stock_symbol) AS distinct_tickers,
                MIN(Date) AS earliest_timestamp,
                MAX(Date) AS latest_timestamp
            FROM '{news_parquet}'
        """).df()
        print("Summary Overview:")
        display(news_summary)
        
        # Breakdown by Ticker
        print("\nTicker Distribution & Date Ranges:")
        news_by_ticker = con.execute(f"""
            SELECT 
                Stock_symbol,
                COUNT(*) AS article_count,
                MIN(Date) AS earliest_date,
                MAX(Date) AS latest_date,
                COUNT(DISTINCT Publisher) AS distinct_publishers
            FROM '{news_parquet}'
            GROUP BY Stock_symbol
            ORDER BY article_count DESC
        """).df()
        display(news_by_ticker)
        
        # Missing / Empty value count across all columns
        print("\nMissing or Empty Field Diagnostics:")
        cols = [col[0] for col in con.execute(f"DESCRIBE SELECT * FROM '{news_parquet}'").fetchall()]
        null_checks = ", ".join([
            f"SUM(CASE WHEN \"{c}\" IS NULL OR \"{c}\" = '' THEN 1 ELSE 0 END) AS \"{c}\""
            for c in cols
        ])
        news_nulls = con.execute(f"SELECT {null_checks} FROM '{news_parquet}'").df()
        display(news_nulls.T.rename(columns={0: 'missing_or_empty_count'}))
        
        # Sample data preview
        print("\nSample Articles (Top 3 Rows):")
        sample_news = con.execute(f"""
            SELECT Stock_symbol, Date, Article_title, Publisher, SUBSTR(Article, 1, 100) AS article_preview
            FROM '{news_parquet}'
            LIMIT 3
        """).df()
        display(sample_news)
    else:
        print(f"❌ Error: News parquet file not found at {news_parquet}")
        
    # ================= 2. PRICES DATASET VALIDATION =================
    print("\n" + "═" * 30 + " 2. FILTERED PRICES DATASET " + "═" * 30)
    if os.path.exists(prices_parquet):
        file_size_mb = os.path.getsize(prices_parquet) / (1024 * 1024)
        print(f"File: {prices_parquet} | Size: {file_size_mb:.2f} MB\n")
        
        # High-level summary
        prices_summary = con.execute(f"""
            SELECT 
                COUNT(*) AS total_rows,
                COUNT(DISTINCT stock_symbol) AS distinct_tickers,
                MIN(date) AS earliest_date,
                MAX(date) AS latest_date
            FROM '{prices_parquet}'
        """).df()
        print("Summary Overview:")
        display(prices_summary)
        
        # Breakdown by Ticker
        print("\nTicker Distribution & Price Range:")
        prices_by_ticker = con.execute(f"""
            SELECT 
                stock_symbol,
                COUNT(*) AS trading_days,
                MIN(date) AS start_date,
                MAX(date) AS end_date,
                ROUND(MIN(low), 2) AS all_time_low,
                ROUND(MAX(high), 2) AS all_time_high,
                ROUND(AVG(close), 2) AS avg_close
            FROM '{prices_parquet}'
            GROUP BY stock_symbol
            ORDER BY stock_symbol ASC
        """).df()
        display(prices_by_ticker)
        
        # Missing / Null diagnostics
        print("\nMissing or Null Values Diagnostics:")
        pcols = [col[0] for col in con.execute(f"DESCRIBE SELECT * FROM '{prices_parquet}'").fetchall()]
        pnull_checks = ", ".join([
            f"SUM(CASE WHEN \"{c}\" IS NULL THEN 1 ELSE 0 END) AS \"{c}\""
            for c in pcols
        ])
        prices_nulls = con.execute(f"SELECT {pnull_checks} FROM '{prices_parquet}'").df()
        display(prices_nulls.T.rename(columns={0: 'null_count'}))
        
        # Sample data preview
        print("\nSample Trading Records (Top 5 Rows):")
        sample_prices = con.execute(f"""
            SELECT * FROM '{prices_parquet}' LIMIT 5
        """).df()
        display(sample_prices)
    else:
        print(f"❌ Error: Prices parquet file not found at {prices_parquet}")
        
    print("\n" + "=" * 80)
    print("🎉 PHASE 1 INGESTION COMPLETE & VALIDATED SUCCESSFULLY!")
    print("=" * 80)

# Execute validation suite
run_validation_suite()

🎯 PHASE 1 COMPREHENSIVE VALIDATION & STATS REPORT

══════════════════════════════ 1. FILTERED NEWS DATASET ══════════════════════════════
File: ./data/filtered_news.parquet | Size: 78.43 MB

Summary Overview:


,total_rows,distinct_tickers,earliest_timestamp,latest_timestamp
0,62458,9,2011-03-03 00:00:00 UTC,2024-01-09 00:00:00 UTC



Ticker Distribution & Date Ranges:


,Stock_symbol,article_count,earliest_date,latest_date,distinct_publishers
0,NVDA,11862,2011-03-03 00:00:00 UTC,2023-12-16 20:00:00 UTC,153
1,TSLA,10587,2019-07-01 00:00:00 UTC,2023-12-16 22:00:00 UTC,59
2,AAPL,9338,2020-03-09 00:00:00 UTC,2023-12-16 22:00:00 UTC,38
3,AMD,9209,2016-12-12 00:00:00 UTC,2024-01-09 00:00:00 UTC,27
4,MSFT,8737,2022-04-26 00:00:00 UTC,2023-12-16 23:02:00 UTC,1
5,AMZN,5060,2020-04-27 00:00:00 UTC,2023-12-16 23:00:00 UTC,37
6,NFLX,3028,2016-08-23 00:00:00 UTC,2020-06-10 12:20:19 UTC,104
7,JPM,2883,2018-09-15 00:00:00 UTC,2020-06-11 06:05:13 UTC,10
8,GOOGL,1754,2018-07-25 00:00:00 UTC,2020-06-10 11:25:13 UTC,75



Missing or Empty Field Diagnostics:


...


Missing or Null Values Diagnostics:


,null_count
date,0.0
open,0.0
high,0.0
low,0.0
close,0.0
adj_close,0.0
volume,0.0
stock_symbol,0.0



Sample Trading Records (Top 5 Rows):


,date,open,high,low,close,adj_close,volume,stock_symbol
0,1980-12-12,0.513393,0.515625,0.513393,0.513393,0.406782,117258400,AAPL
1,1980-12-15,0.488839,0.488839,0.486607,0.486607,0.385558,43971200,AAPL
2,1980-12-16,0.453125,0.453125,0.450893,0.450893,0.357260,26432000,AAPL
3,1980-12-17,0.462054,0.464286,0.462054,0.462054,0.366103,21610400,AAPL
4,1980-12-18,0.475446,0.477679,0.475446,0.475446,0.376715,18362400,AAPL



🎉 PHASE 1 INGESTION COMPLETE & VALIDATED SUCCESSFULLY!


## 💾 Step 6: Persistent Backup (Google Drive & Local ZIP)

Before the Google Colab T4 session disconnects, persist both processed `.parquet` files:
1. **Google Drive Sync:** Mounts `/content/drive` via `google.colab.drive.mount` and copies files into `/content/drive/MyDrive/NEXUS_Stock_AI/data/` using `shutil.copy2` (preserving file metadata).
2. **Local ZIP Archive:** Compresses `./data/` into `./nexus_data_backup.zip` so you can right-click and download it via the VS Code or Colab file explorer.
3. **Confirmation Report:** Prints exact file paths and sizes confirming safe preservation.

In [7]:
import os
import shutil
import zipfile
import time

DATA_DIR = "./data"
GDRIVE_DEST_DIR = "/content/drive/MyDrive/NEXUS_Stock_AI/data"
LOCAL_ZIP_PATH = "./nexus_data_backup.zip"
TARGET_FILES = ["filtered_news.parquet", "filtered_prices.parquet"]

print("=" * 80)
print("🚀 BACKING UP PHASE 1 ARTIFACTS")
print("=" * 80)

# 1. Mount Google Drive
try:
    from google.colab import drive
    print("\n[1/3] Mounting Google Drive at /content/drive...")
    drive.mount("/content/drive")
    print("  ✓ Google Drive mounted successfully.")
except Exception as e:
    print(f"  Drive mount note: {e}")

# 2. Sync files to Google Drive
print("\n[2/3] Copying artifacts to Google Drive...")
gdrive_copied = []
if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(GDRIVE_DEST_DIR, exist_ok=True)
    print(f"  Target directory: {GDRIVE_DEST_DIR}")
    for fname in TARGET_FILES:
        src = os.path.join(DATA_DIR, fname)
        dst = os.path.join(GDRIVE_DEST_DIR, fname)
        if os.path.exists(src):
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / (1024 * 1024)
            gdrive_copied.append((dst, size_mb))
            print(f"  ✓ Copied: {fname} -> {dst} ({size_mb:.2f} MB)")
        else:
            print(f"  ⚠️ Source file not found: {src}")
else:
    print("  ℹ️ /content/drive/MyDrive not available. Skipping Google Drive copy.")

# 3. Create local ZIP archive in root directory
print(f"\n[3/3] Creating local ZIP archive: {LOCAL_ZIP_PATH}...")
archived_files = []
if os.path.exists(DATA_DIR):
    with zipfile.ZipFile(LOCAL_ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(DATA_DIR):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.join("data", os.path.relpath(file_path, DATA_DIR))
                zipf.write(file_path, arcname)
                size_mb = os.path.getsize(file_path) / (1024 * 1024)
                archived_files.append((arcname, size_mb))
                print(f"  + Archived: {arcname} ({size_mb:.2f} MB)")

zip_size_mb = os.path.getsize(LOCAL_ZIP_PATH) / (1024 * 1024) if os.path.exists(LOCAL_ZIP_PATH) else 0

# 4. Confirmation Report
print("\n" + "=" * 80)
print("✅ PHASE 1 ARTIFACTS SAFELY BACKED UP!")
print("=" * 80)
if gdrive_copied:
    print("📁 Google Drive Persistent Storage:")
    for dst, size_mb in gdrive_copied:
        print(f"   • {dst} ({size_mb:.2f} MB)")
if os.path.exists(LOCAL_ZIP_PATH):
    print(f"\n📦 Local ZIP Archive (Downloadable via VS Code / Colab):")
    print(f"   • Exact Path: {os.path.abspath(LOCAL_ZIP_PATH)} ({zip_size_mb:.2f} MB)")
    print(f"   • Contained Files:")
    for arcname, size_mb in archived_files:
        print(f"       - {arcname} ({size_mb:.2f} MB)")
    print(f"\n💡 How to download: In VS Code file explorer or Colab sidebar, right-click nexus_data_backup.zip -> Download.")
print("=" * 80)


Mounted at /content/drive
  ✓ Google Drive mounted successfully.

[2/3] Copying artifacts to Google Drive...
  Target directory: /content/drive/MyDrive/NEXUS_Stock_AI/data
  ✓ Copied: filtered_news.parquet -> /content/drive/MyDrive/NEXUS_Stock_AI/data/filtered_news.parquet (78.43 MB)
  ✓ Copied: filtered_prices.parquet -> /content/drive/MyDrive/NEXUS_Stock_AI/data/filtered_prices.parquet (1.74 MB)

[3/3] Creating local ZIP archive: ./nexus_data_backup.zip...
  + Archived: data/filtered_news.parquet (78.43 MB)
  + Archived: data/filtered_prices.parquet (1.74 MB)

✅ PHASE 1 ARTIFACTS SAFELY BACKED UP!
📁 Google Drive Persistent Storage:
   • /content/drive/MyDrive/NEXUS_Stock_AI/data/filtered_news.parquet (78.43 MB)
   • /content/drive/MyDrive/NEXUS_Stock_AI/data/filtered_prices.parquet (1.74 MB)

📦 Local ZIP Archive (Downloadable via VS Code / Colab):
   • Exact Path: /content/nexus_data_backup.zip (80.07 MB)
   • Contained Files:
       - data/filtered_news.parquet (78.43 MB)
       - d